# DogHeart X-ray Classification Homework

**Student**: 2025120019 王咏琪  
**Task**: classify canine chest X-rays into `Large`, `Normal`, and `Small`.

This notebook is refactored as a self-contained, readable homework submission. The code is split by responsibility inside the notebook: configuration, data loading, model definition, metrics, training loop, validation, and prediction. External `.py` files in the GitHub repository contain the same cleaned implementation, but the core workflow is also visible here for grading.

- GitHub repository: [https://github.com/wang935/homework](https://github.com/wang935/homework)
- Notebook: [https://github.com/wang935/homework/blob/main/2025120019%2B%E7%8E%8B%E5%92%8F%E7%90%AA.ipynb](https://github.com/wang935/homework/blob/main/2025120019%2B%E7%8E%8B%E5%92%8F%E7%90%AA.ipynb)
- Final checkpoint: [https://github.com/wang935/homework/blob/main/models/final_model.pt](https://github.com/wang935/homework/blob/main/models/final_model.pt)
- Prediction CSV: [https://github.com/wang935/homework/blob/main/outputs/results.csv](https://github.com/wang935/homework/blob/main/outputs/results.csv)
- Validation metrics: [https://github.com/wang935/homework/blob/main/outputs/convnext_tiny_final/valid_metrics.json](https://github.com/wang935/homework/blob/main/outputs/convnext_tiny_final/valid_metrics.json)
- Paper link: [https://www.researchgate.net/publication/407236704_An_Architecture_Benchmark_for_Canine_Heart-Size_Classification_on_DogHeart](https://www.researchgate.net/publication/407236704_An_Architecture_Benchmark_for_Canine_Heart-Size_Classification_on_DogHeart)
- DOI: `10.13140/RG.2.2.28742.23369`
- Report PDF: [https://github.com/wang935/homework/blob/main/report/main_final.pdf](https://github.com/wang935/homework/blob/main/report/main_final.pdf)


## 1. Configuration

All path and hyperparameter choices are centralized here instead of being scattered through later cells.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ProjectPaths:
    root: Path = Path(r"C:\Users\wang\Desktop\Homework")
    data_root: Path = Path(r"C:\Users\wang\Desktop\Homework\Homework\Dog_Heart\Dog_Heart")
    checkpoint: Path = Path(r"C:\Users\wang\Desktop\Homework\models\final_model.pt")
    output_csv: Path = Path(r"C:\Users\wang\Desktop\Homework\outputs\results.csv")
    metrics_json: Path = Path(r"C:\Users\wang\Desktop\Homework\outputs\convnext_tiny_final\valid_metrics_recheck.json")


@dataclass(frozen=True)
class TrainConfig:
    architecture: str = "convnext_tiny"
    pretrained: bool = True
    image_size: int = 224
    resize_mode: str = "stretch"
    aug_strength: str = "strong"
    epochs: int = 80
    batch_size: int = 48
    learning_rate: float = 2e-4
    weight_decay: float = 0.1
    label_smoothing: float = 0.1
    mixup_alpha: float = 0.4
    patience: int = 12
    seed: int = 20260525
    num_workers: int = 0
    device: str = "auto"


PATHS = ProjectPaths()
CONFIG = TrainConfig()
print(PATHS)
print(CONFIG)


## 2. Model Definition

The model cell exposes a compact custom CNN and a `build_model` factory for transfer-learning backbones. The submitted checkpoint uses `convnext_tiny`, while `DogHeartCNN` satisfies the custom-CNN requirement.


In [ ]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass
from typing import Any

import torch
from torch import nn
from torchvision import models


class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, dropout: float = 0.0) -> None:
        super().__init__()
        self.conv1 = ConvBNAct(in_channels, out_channels, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        x = self.conv1(x)
        x = self.drop(x)
        x = self.conv2(x)
        return self.act(x + residual)


class DogHeartCNN(nn.Module):
    """A compact custom CNN for dog chest X-ray heart-size classification."""

    def __init__(self, num_classes: int = 3, dropout: float = 0.25) -> None:
        super().__init__()
        self.features = nn.Sequential(
            ConvBNAct(3, 32, stride=2),
            ConvBNAct(32, 48),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            ResidualBlock(48, 64, stride=1, dropout=0.05),
            ResidualBlock(64, 96, stride=2, dropout=0.08),
            ResidualBlock(96, 128, stride=1, dropout=0.08),
            ResidualBlock(128, 192, stride=2, dropout=0.10),
            ResidualBlock(192, 256, stride=1, dropout=0.10),
            ResidualBlock(256, 320, stride=2, dropout=0.12),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(320, 128),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


@dataclass(frozen=True)
class TorchvisionModelSpec:
    builder: Callable[..., nn.Module]
    default_weights: Any
    replace_head: Callable[[nn.Module, int, float], None]


def _dropout_classifier(in_features: int, num_classes: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes),
    )


def _replace_fc_with_dropout(model: nn.Module, num_classes: int, dropout: float) -> None:
    in_features = model.fc.in_features
    model.fc = _dropout_classifier(in_features, num_classes, dropout)


def _replace_fc(model: nn.Module, num_classes: int, dropout: float) -> None:
    del dropout
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)


def _replace_classifier_with_dropout(model: nn.Module, num_classes: int, dropout: float) -> None:
    in_features = model.classifier.in_features
    model.classifier = _dropout_classifier(in_features, num_classes, dropout)


def _replace_classifier_tail_with_dropout(model: nn.Module, num_classes: int, dropout: float) -> None:
    in_features = model.classifier[-1].in_features
    model.classifier = _dropout_classifier(in_features, num_classes, dropout)


def _replace_classifier_tail(model: nn.Module, num_classes: int, dropout: float) -> None:
    del dropout
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)


def _replace_head(model: nn.Module, num_classes: int, dropout: float) -> None:
    del dropout
    in_features = model.head.in_features
    model.head = nn.Linear(in_features, num_classes)


TORCHVISION_MODEL_SPECS: dict[str, TorchvisionModelSpec] = {
    "resnet18": TorchvisionModelSpec(models.resnet18, models.ResNet18_Weights.DEFAULT, _replace_fc_with_dropout),
    "resnet34": TorchvisionModelSpec(models.resnet34, models.ResNet34_Weights.DEFAULT, _replace_fc_with_dropout),
    "resnet50": TorchvisionModelSpec(models.resnet50, models.ResNet50_Weights.DEFAULT, _replace_fc_with_dropout),
    "densenet121": TorchvisionModelSpec(
        models.densenet121,
        models.DenseNet121_Weights.DEFAULT,
        _replace_classifier_with_dropout,
    ),
    "efficientnet_b0": TorchvisionModelSpec(
        models.efficientnet_b0,
        models.EfficientNet_B0_Weights.DEFAULT,
        _replace_classifier_tail_with_dropout,
    ),
    "efficientnet_b2": TorchvisionModelSpec(
        models.efficientnet_b2,
        models.EfficientNet_B2_Weights.DEFAULT,
        _replace_classifier_tail_with_dropout,
    ),
    "efficientnet_v2_s": TorchvisionModelSpec(
        models.efficientnet_v2_s,
        models.EfficientNet_V2_S_Weights.DEFAULT,
        _replace_classifier_tail_with_dropout,
    ),
    "convnext_tiny": TorchvisionModelSpec(
        models.convnext_tiny,
        models.ConvNeXt_Tiny_Weights.DEFAULT,
        _replace_classifier_tail,
    ),
    "convnext_small": TorchvisionModelSpec(
        models.convnext_small,
        models.ConvNeXt_Small_Weights.DEFAULT,
        _replace_classifier_tail,
    ),
    "mobilenet_v3_large": TorchvisionModelSpec(
        models.mobilenet_v3_large,
        models.MobileNet_V3_Large_Weights.DEFAULT,
        _replace_classifier_tail,
    ),
    "regnet_y_400mf": TorchvisionModelSpec(
        models.regnet_y_400mf,
        models.RegNet_Y_400MF_Weights.DEFAULT,
        _replace_fc,
    ),
    "regnet_y_800mf": TorchvisionModelSpec(
        models.regnet_y_800mf,
        models.RegNet_Y_800MF_Weights.DEFAULT,
        _replace_fc,
    ),
    "swin_t": TorchvisionModelSpec(models.swin_t, models.Swin_T_Weights.DEFAULT, _replace_head),
}

SUPPORTED_ARCHITECTURES: tuple[str, ...] = ("custom", *TORCHVISION_MODEL_SPECS.keys())


def _resolve_weights(default_weights: Any, pretrained: bool) -> Any:
    return default_weights if pretrained else None


def build_model(
    num_classes: int = 3,
    dropout: float = 0.25,
    architecture: str = "custom",
    pretrained: bool = False,
) -> nn.Module:
    architecture = architecture.lower()
    if architecture == "custom":
        return DogHeartCNN(num_classes=num_classes, dropout=dropout)

    spec = TORCHVISION_MODEL_SPECS.get(architecture)
    if spec is None:
        supported = ", ".join(SUPPORTED_ARCHITECTURES)
        raise ValueError(f"Unknown architecture: {architecture}. Supported architectures: {supported}")

    model = spec.builder(weights=_resolve_weights(spec.default_weights, pretrained))
    spec.replace_head(model, num_classes, dropout)
    return model


## 3. Data Loading and Preprocessing

The dataset code fixes class order as `Large=0`, `Normal=1`, `Small=2`; this avoids accidental label remapping between training and submission.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Callable

from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset
from torchvision import datasets, transforms


CLASS_NAMES = ["Large", "Normal", "Small"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
ImageSize = int | tuple[int, int]


def natural_key(path: Path) -> list[object]:
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name)]


class TestImageDataset(Dataset):
    """Test dataset returning image tensor and original file name."""

    def __init__(self, image_dir: str | Path, transform: Callable | None = None) -> None:
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.paths = sorted(
            [p for p in self.image_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}],
            key=natural_key,
        )
        if not self.paths:
            raise FileNotFoundError(f"No images found in {self.image_dir}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, str]:
        path = self.paths[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, path.name


class ResizePad:
    def __init__(self, size: ImageSize, fill: int = 0) -> None:
        self.size = _resize_size(size)
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        target_h, target_w = self.size
        image = ImageOps.contain(image, (target_w, target_h), method=Image.Resampling.BILINEAR)
        canvas = Image.new(image.mode, (target_w, target_h), color=self.fill)
        left = (target_w - image.width) // 2
        top = (target_h - image.height) // 2
        canvas.paste(image, (left, top))
        return canvas


def _resize_size(image_size: ImageSize) -> tuple[int, int]:
    return image_size if isinstance(image_size, tuple) else (image_size, image_size)


def _resize_transform(image_size: ImageSize, resize_mode: str) -> transforms.Resize | ResizePad:
    if resize_mode == "stretch":
        return transforms.Resize(_resize_size(image_size))
    if resize_mode == "pad":
        return ResizePad(image_size)
    raise ValueError(f"Unknown resize mode: {resize_mode}")


def train_transform(image_size: ImageSize = 224, strength: str = "standard", resize_mode: str = "stretch") -> transforms.Compose:
    normalize = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    resize = _resize_transform(image_size, resize_mode)
    if strength == "none":
        return eval_transform(image_size, resize_mode)
    if strength == "light":
        return transforms.Compose(
            [
                resize,
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=3),
                transforms.ColorJitter(brightness=0.05, contrast=0.05),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if strength == "randaug":
        return transforms.Compose(
            [
                resize,
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandAugment(num_ops=2, magnitude=9),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if strength == "strong":
        return transforms.Compose(
            [
                resize,
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
                transforms.RandAugment(num_ops=2, magnitude=7),
                transforms.ToTensor(),
                normalize,
            ]
        )
    if strength != "standard":
        raise ValueError(f"Unknown augmentation strength: {strength}")
    return transforms.Compose(
        [
            resize,
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=8),
            transforms.ColorJitter(brightness=0.12, contrast=0.12),
            transforms.ToTensor(),
            normalize,
        ]
    )


def eval_transform(image_size: ImageSize = 224, resize_mode: str = "stretch") -> transforms.Compose:
    return transforms.Compose(
        [
            _resize_transform(image_size, resize_mode),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )


def make_imagefolder(root: str | Path, transform: Callable | None = None) -> datasets.ImageFolder:
    dataset = datasets.ImageFolder(str(root), transform=transform)
    if dataset.classes != CLASS_NAMES:
        raise ValueError(
            f"Unexpected class order {dataset.classes}. Expected {CLASS_NAMES}; "
            "the output CSV depends on this fixed mapping."
        )
    return dataset


## 4. Metrics and Device Utilities

Metrics are small pure functions so that validation reporting is reproducible and independent of notebook display output.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import torch


def confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int) -> torch.Tensor:
    matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for true, pred in zip(y_true.view(-1), y_pred.view(-1)):
        matrix[int(true), int(pred)] += 1
    return matrix


def classification_metrics(matrix: torch.Tensor, class_names: list[str]) -> dict:
    total = matrix.sum().item()
    correct = matrix.diag().sum().item()
    per_class = {}
    f1_values = []
    for idx, name in enumerate(class_names):
        tp = matrix[idx, idx].item()
        fp = matrix[:, idx].sum().item() - tp
        fn = matrix[idx, :].sum().item() - tp
        support = matrix[idx, :].sum().item()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1_values.append(f1)
        per_class[name] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    return {
        "accuracy": correct / total if total else 0.0,
        "macro_f1": sum(f1_values) / len(f1_values) if f1_values else 0.0,
        "confusion_matrix": matrix.tolist(),
        "per_class": per_class,
    }


def save_metrics(metrics: dict, output_path: str | Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")


In [ ]:
from __future__ import annotations

import torch


def resolve_device(requested: str = "auto") -> torch.device:
    if requested != "auto":
        return torch.device(requested)

    if not torch.cuda.is_available():
        return torch.device("cpu")

    capability = torch.cuda.get_device_capability()
    arch = f"sm_{capability[0]}{capability[1]}"
    supported_arches = set(torch.cuda.get_arch_list())
    if arch not in supported_arches:
        print(
            f"CUDA device capability {arch} is not supported by this PyTorch build "
            f"({sorted(supported_arches)}). Falling back to CPU."
        )
        return torch.device("cpu")

    return torch.device("cuda")


## 5. Refactored Training Loop

The original notebook-style script is reorganized into reusable functions: `make_loaders`, `run_epoch`, and `train_model`. Training is not automatically launched when the notebook opens; call `train_model()` to retrain.


In [ ]:
import csv
import json
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def make_loaders(config: TrainConfig, paths: ProjectPaths) -> tuple[DataLoader, DataLoader, list[float]]:
    train_ds = make_imagefolder(
        paths.data_root / "Train",
        train_transform(config.image_size, config.aug_strength, config.resize_mode),
    )
    valid_ds = make_imagefolder(
        paths.data_root / "Valid",
        eval_transform(config.image_size, config.resize_mode),
    )

    counts = torch.bincount(torch.tensor(train_ds.targets), minlength=len(CLASS_NAMES)).float()
    class_weights = (counts.sum() / (len(CLASS_NAMES) * counts)).tolist()
    sample_weights = torch.tensor([class_weights[target] for target in train_ds.targets], dtype=torch.double)

    loader_kwargs = {
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "pin_memory": torch.cuda.is_available(),
    }
    train_loader = DataLoader(
        train_ds,
        sampler=WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True),
        **loader_kwargs,
    )
    valid_loader = DataLoader(valid_ds, shuffle=False, **loader_kwargs)
    return train_loader, valid_loader, class_weights


def mixup_batch(images: torch.Tensor, targets: torch.Tensor, alpha: float) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    lam = max(lam, 1.0 - lam)
    index = torch.randperm(images.size(0), device=images.device)
    mixed_images = lam * images + (1.0 - lam) * images[index]
    return mixed_images, targets, targets[index], lam


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
    mixup_alpha: float | None = None,
) -> tuple[float, torch.Tensor, torch.Tensor]:
    is_train = optimizer is not None
    model.train(is_train)
    losses: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    all_preds: list[torch.Tensor] = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        use_mixup = is_train and mixup_alpha is not None and mixup_alpha > 0
        if use_mixup:
            images, targets_a, targets_b, lam = mixup_batch(images, targets, mixup_alpha)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            if use_mixup:
                loss = lam * criterion(logits, targets_a) + (1.0 - lam) * criterion(logits, targets_b)
                metric_targets = targets_a
            else:
                loss = criterion(logits, targets)
                metric_targets = targets

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        losses.append(loss.detach().cpu() * images.size(0))
        all_targets.append(metric_targets.detach().cpu())
        all_preds.append(logits.detach().argmax(dim=1).cpu())

    avg_loss = torch.stack(losses).sum().item() / len(loader.dataset)
    return avg_loss, torch.cat(all_targets), torch.cat(all_preds)


def train_model(config: TrainConfig = CONFIG, paths: ProjectPaths = PATHS) -> dict:
    set_seed(config.seed)
    device = resolve_device(config.device)
    train_loader, valid_loader, class_weights = make_loaders(config, paths)

    model = build_model(
        num_classes=len(CLASS_NAMES),
        architecture=config.architecture,
        pretrained=config.pretrained,
    ).to(device)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32, device=device),
        label_smoothing=config.label_smoothing,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    best_acc = -1.0
    best_state = None
    history: list[dict] = []
    no_improve = 0

    for epoch in range(1, config.epochs + 1):
        train_loss, train_true, train_pred = run_epoch(
            model, train_loader, criterion, device, optimizer, config.mixup_alpha
        )
        valid_loss, valid_true, valid_pred = run_epoch(model, valid_loader, criterion, device)
        scheduler.step()

        valid_matrix = confusion_matrix(valid_true, valid_pred, len(CLASS_NAMES))
        valid_metrics = classification_metrics(valid_matrix, CLASS_NAMES)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": float((train_true == train_pred).float().mean()),
            "valid_loss": valid_loss,
            "valid_acc": valid_metrics["accuracy"],
            "valid_macro_f1": valid_metrics["macro_f1"],
        }
        history.append(row)
        print(row)

        if valid_metrics["accuracy"] > best_acc:
            best_acc = valid_metrics["accuracy"]
            best_state = model.state_dict()
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= config.patience:
                break

    return {"best_accuracy": best_acc, "best_state": best_state, "history": history}


## 6. Evaluation and Prediction

Validation and submission generation are also functions rather than copied code blocks. This makes it clear which checkpoint, transform, and class mapping are used.


In [ ]:
import csv

import torch
from torch import nn
from torch.utils.data import DataLoader


@torch.no_grad()
def evaluate_checkpoint(
    checkpoint_path: Path = PATHS.checkpoint,
    data_root: Path = PATHS.data_root,
    output_path: Path = PATHS.metrics_json,
    device_name: str = CONFIG.device,
    batch_size: int = 64,
    num_workers: int = 0,
) -> dict:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    image_size = checkpoint.get("image_size", CONFIG.image_size)
    resize_mode = checkpoint.get("resize_mode", CONFIG.resize_mode)
    device = resolve_device(device_name)

    dataset = make_imagefolder(data_root / "Valid", eval_transform(image_size, resize_mode))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    model = build_model(
        num_classes=len(CLASS_NAMES),
        architecture=checkpoint.get("architecture", CONFIG.architecture),
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    y_true: list[torch.Tensor] = []
    y_pred: list[torch.Tensor] = []
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        logits = model(images)
        total_loss += criterion(logits, targets).item() * images.size(0)
        y_true.append(targets.cpu())
        y_pred.append(logits.argmax(dim=1).cpu())

    true = torch.cat(y_true)
    pred = torch.cat(y_pred)
    metrics = classification_metrics(confusion_matrix(true, pred, len(CLASS_NAMES)), CLASS_NAMES)
    metrics["loss"] = total_loss / len(dataset)
    save_metrics(metrics, output_path)
    print(json.dumps(metrics, indent=2))
    return metrics


@torch.no_grad()
def predict_test_csv(
    checkpoint_path: Path = PATHS.checkpoint,
    test_dir: Path = PATHS.data_root / "Test" / "Images",
    output_csv: Path = PATHS.output_csv,
    device_name: str = CONFIG.device,
    batch_size: int = 64,
    num_workers: int = 0,
) -> list[tuple[str, int]]:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    image_size = checkpoint.get("image_size", CONFIG.image_size)
    resize_mode = checkpoint.get("resize_mode", CONFIG.resize_mode)
    device = resolve_device(device_name)

    dataset = TestImageDataset(test_dir, eval_transform(image_size, resize_mode))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    model = build_model(
        num_classes=len(CLASS_NAMES),
        architecture=checkpoint.get("architecture", CONFIG.architecture),
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    rows: list[tuple[str, int]] = []
    for images, names in loader:
        logits = model(images.to(device))
        labels = logits.argmax(dim=1).cpu().tolist()
        rows.extend((name, int(label)) for name, label in zip(names, labels))

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    with output_csv.open("w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)
    print(f"wrote {len(rows)} rows to {output_csv}")
    return rows


def validate_submission_csv(csv_path: Path = PATHS.output_csv, test_dir: Path = PATHS.data_root / "Test" / "Images") -> None:
    expected = {path.name for path in sorted(test_dir.iterdir()) if path.suffix.lower() in {".png", ".jpg", ".jpeg"}}
    with csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    names = [row[0] for row in rows]
    labels = [row[1] for row in rows]
    assert len(rows) == len(expected), (len(rows), len(expected))
    assert set(names) == expected
    assert len(names) == len(set(names))
    assert set(labels) <= {"0", "1", "2"}
    print(f"OK: {csv_path} has {len(rows)} valid rows.")


## 7. Final Results

| Metric | Value |
|---|---:|
| Validation accuracy | **77.5%** |
| Validation macro-F1 | **79.4%** |
| Hidden test accuracy | **77.25%** |

Validation confusion matrix (`rows=true`, `columns=pred`):

```text
[[55, 21, 0], [15, 68, 8], [0, 1, 32]]
```


In [ ]:
# Reproduce the final validation metrics from the uploaded checkpoint.
# Expected: accuracy=0.775, macro_f1=0.7938.
final_metrics = evaluate_checkpoint()


In [ ]:
# Regenerate the hidden-test prediction CSV and validate its format.
# Uncomment the first line if you want to overwrite outputs/results.csv.
# predict_test_csv()
validate_submission_csv()


## 8. Comparison with RVT and Submission Links

RVT uses anatomical key-point supervision and reports stronger performance. This submission is a classification-only ConvNeXt Tiny pipeline, so it is framed as a reproducible coursework baseline rather than a claim to outperform RVT.

- Paper: [https://www.researchgate.net/publication/407236704_An_Architecture_Benchmark_for_Canine_Heart-Size_Classification_on_DogHeart](https://www.researchgate.net/publication/407236704_An_Architecture_Benchmark_for_Canine_Heart-Size_Classification_on_DogHeart)
- DOI: `10.13140/RG.2.2.28742.23369`
- GitHub repository: [https://github.com/wang935/homework](https://github.com/wang935/homework)
- GitHub checkpoint: [https://github.com/wang935/homework/blob/main/models/final_model.pt](https://github.com/wang935/homework/blob/main/models/final_model.pt)
- GitHub prediction CSV: [https://github.com/wang935/homework/blob/main/outputs/results.csv](https://github.com/wang935/homework/blob/main/outputs/results.csv)
- Report PDF: [https://github.com/wang935/homework/blob/main/report/main_final.pdf](https://github.com/wang935/homework/blob/main/report/main_final.pdf)
